# Loan Approval Model Training

This notebook loads and validates the loan dataset, creates a preprocessing pipeline, trains a random-forest classifier, evaluates it on an untouched holdout set, and saves the complete model. Run the cells from top to bottom.

In [ ]:
# 1. Import the required libraries
import json
import warnings
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 2. Configure paths and training settings

Change `TEST_SIZE`, `CV_FOLDS`, or `RANDOM_STATE` here if you want to experiment. The notebook expects to be run from the project folder.

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "dataset" / "loan_approval_dataset.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "loan_approval_model.joblib"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics.json"

TEST_SIZE = 0.20
CV_FOLDS = 5
RANDOM_STATE = 42

TARGET_COLUMN = "loan_status"
ID_COLUMN = "loan_id"
LABELS = ["Rejected", "Approved"]

NUMERIC_FEATURES = [
    "no_of_dependents",
    "income_annum",
    "loan_amount",
    "loan_term",
    "cibil_score",
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]
CATEGORICAL_FEATURES = ["education", "self_employed"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
ASSET_FEATURES = [
    "residential_assets_value",
    "commercial_assets_value",
    "luxury_assets_value",
    "bank_asset_value",
]

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Open the project folder in VS Code first."
    )

print(f"Project folder: {PROJECT_ROOT}")

## 3. Load and clean the dataset

The CSV contains spaces after its commas, so `skipinitialspace=True` and string trimming normalize the column names and categorical values.

In [ ]:
data = pd.read_csv(DATA_PATH, skipinitialspace=True)
data.columns = data.columns.str.strip()

required_columns = set(FEATURES + [ID_COLUMN, TARGET_COLUMN])
missing_columns = sorted(required_columns.difference(data.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

data = data[FEATURES + [ID_COLUMN, TARGET_COLUMN]].copy()

for column in CATEGORICAL_FEATURES + [TARGET_COLUMN]:
    data[column] = data[column].astype("string").str.strip()

for column in NUMERIC_FEATURES + [ID_COLUMN]:
    data[column] = pd.to_numeric(data[column], errors="raise")

print(f"Dataset shape: {data.shape}")
display(data.head())

## 4. Validate and explore the data

This checks the IDs, missing data, target labels, category values, duplicates, class balance, and numerical ranges. The 28 negative residential-asset entries are reported but retained so the notebook does not silently rewrite the source data.

In [ ]:
if data.empty:
    raise ValueError("Dataset contains no rows.")
if data[ID_COLUMN].isna().any():
    raise ValueError(f"{ID_COLUMN} contains missing values.")
if data[ID_COLUMN].duplicated().any():
    raise ValueError(f"{ID_COLUMN} must be unique.")
if data[FEATURES + [TARGET_COLUMN]].isna().any().any():
    missing_values = data[FEATURES + [TARGET_COLUMN]].isna().sum()
    missing_values = missing_values[missing_values.gt(0)].to_dict()
    raise ValueError(f"Dataset contains missing values: {missing_values}")

observed_labels = set(data[TARGET_COLUMN].unique())
if observed_labels != set(LABELS):
    raise ValueError(f"Expected target labels {LABELS}; found {sorted(observed_labels)}")

allowed_categories = {
    "education": {"Graduate", "Not Graduate"},
    "self_employed": {"Yes", "No"},
}
for column, allowed_values in allowed_categories.items():
    unexpected = sorted(set(data[column].unique()).difference(allowed_values))
    if unexpected:
        raise ValueError(f"Unexpected values in {column}: {unexpected}")

negative_assets = {
    column: int(data[column].lt(0).sum())
    for column in ASSET_FEATURES
    if data[column].lt(0).any()
}
if negative_assets:
    warnings.warn(f"Negative asset values retained: {negative_assets}")

target_counts = data[TARGET_COLUMN].value_counts()
print(f"Missing values: {int(data.isna().sum().sum())}")
print(f"Duplicate rows: {int(data.duplicated().sum())}")
display(pd.DataFrame({
    "count": target_counts,
    "percentage": (target_counts / len(data) * 100).round(2),
}))
display(data[NUMERIC_FEATURES].describe().T)

## 5. Select features and create a holdout set

`loan_id` is excluded because it identifies a record rather than describing an applicant. Stratification preserves the Approved/Rejected ratio in both partitions.

In [ ]:
X = data[FEATURES]
y = data[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Holdout test rows: {len(X_test)}")

## 6. Build preprocessing and model pipelines

Numerical values use median imputation. Categorical values use the most frequent category and one-hot encoding. The preprocessing and classifier are combined so the saved model accepts raw applicant rows.

In [ ]:
numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

classifier = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ]
)

model

## 7. Cross-validate on the training partition

Five-fold validation estimates how consistently the model performs without looking at the final holdout set.

In [ ]:
cross_validator = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cross_validator,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "roc_auc": "roc_auc",
    },
    n_jobs=1,
)

cv_summary = pd.DataFrame({
    metric: {
        "mean": cv_scores[f"test_{metric}"].mean(),
        "standard_deviation": cv_scores[f"test_{metric}"].std(),
    }
    for metric in ("accuracy", "macro_f1", "roc_auc")
}).T
display(cv_summary.round(4))

## 8. Train and evaluate on the untouched holdout set

Accuracy shows total correctness, macro-F1 weights both classes equally, and ROC-AUC measures ranking quality across probability thresholds.

In [ ]:
model.fit(X_train, y_train)
predictions = model.predict(X_test)

class_names = list(model.classes_)
approved_index = class_names.index("Approved")
approval_probabilities = model.predict_proba(X_test)[:, approved_index]
binary_test_target = y_test.eq("Approved").astype(int)

accuracy = accuracy_score(y_test, predictions)
balanced_accuracy = balanced_accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average="macro")
roc_auc = roc_auc_score(binary_test_target, approval_probabilities)
confusion = confusion_matrix(y_test, predictions, labels=LABELS)
report = classification_report(
    y_test, predictions, labels=LABELS, output_dict=True, zero_division=0
)

display(pd.Series({
    "accuracy": accuracy,
    "balanced_accuracy": balanced_accuracy,
    "macro_f1": macro_f1,
    "roc_auc": roc_auc,
}, name="holdout_score").round(4))

display(pd.DataFrame(confusion, index=LABELS, columns=LABELS))
print(classification_report(
    y_test, predictions, labels=LABELS, digits=4, zero_division=0
))

## 9. Inspect feature importance

Random-forest importance estimates how much each encoded feature contributes to reducing prediction error within its trees.

In [ ]:
transformed_features = model.named_steps["preprocessor"].get_feature_names_out()
feature_importance = sorted(
    zip(transformed_features, classifier.feature_importances_, strict=True),
    key=lambda item: item[1],
    reverse=True,
)

importance_table = pd.DataFrame(
    feature_importance, columns=["feature", "importance"]
).head(10)
display(importance_table)

## 10. Save the model and metrics

The `.joblib` file stores both preprocessing and prediction. The JSON file stores the data profile and evaluation results.

In [ ]:
cross_validation_metrics = {
    "folds": CV_FOLDS,
    **{
        metric: {
            "mean": float(cv_scores[f"test_{metric}"].mean()),
            "standard_deviation": float(cv_scores[f"test_{metric}"].std()),
        }
        for metric in ("accuracy", "macro_f1", "roc_auc")
    },
}

metrics = {
    "data": {
        "rows": int(len(data)),
        "input_features": len(FEATURES),
        "duplicate_rows": int(data.duplicated().sum()),
        "target_distribution": {
            str(label): int(count) for label, count in target_counts.items()
        },
        "negative_asset_values": negative_assets,
    },
    "split": {
        "training_rows": int(len(X_train)),
        "test_rows": int(len(X_test)),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
    },
    "model": {
        "type": "RandomForestClassifier",
        "features": FEATURES,
        "target": TARGET_COLUMN,
        "classes": class_names,
    },
    "cross_validation_on_training_partition": cross_validation_metrics,
    "holdout_test": {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "macro_f1": float(macro_f1),
        "roc_auc": float(roc_auc),
        "confusion_matrix": {
            "label_order": LABELS,
            "values": confusion.tolist(),
        },
        "classification_report": report,
    },
    "top_feature_importance": [
        {"feature": str(feature), "importance": float(importance)}
        for feature, importance in feature_importance[:10]
    ],
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_PATH)
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Model saved to: {MODEL_PATH}")
print(f"Metrics saved to: {METRICS_PATH}")

## 11. Try one prediction

This final optional cell reloads the saved artifact and predicts one example applicant.

In [ ]:
saved_model = joblib.load(MODEL_PATH)
example_applicant = pd.DataFrame([{
    "no_of_dependents": 2,
    "education": "Graduate",
    "self_employed": "No",
    "income_annum": 5_000_000,
    "loan_amount": 12_000_000,
    "loan_term": 10,
    "cibil_score": 750,
    "residential_assets_value": 5_000_000,
    "commercial_assets_value": 2_000_000,
    "luxury_assets_value": 8_000_000,
    "bank_asset_value": 3_000_000,
}])

predicted_label = saved_model.predict(example_applicant)[0]
approved_index = list(saved_model.classes_).index("Approved")
approval_probability = saved_model.predict_proba(example_applicant)[0, approved_index]

print(f"Prediction: {predicted_label}")
print(f"Approval probability: {approval_probability:.2%}")